# Download AutoPET-III (PSMA-PET-CT-Lesions) DICOM to Google Drive

Downloads PSMA-PET-CT-Lesions DICOM data from TCIA directly to Google Drive
using the TCIA REST API, bypassing local storage and NBIA Data Retriever.

**Size:** ~117 GB (1,791 series)
**License:** CC BY 4.0
**Pre-registration:** https://doi.org/10.17605/OSF.IO/4KAZN

**Storage format (updated 2026-04-26):** New downloads are saved as `{uid}.zip` on Drive (single large write per series). Earlier downloads (~678 series before the update) are stored as extracted directories — both formats are valid and the processing notebook will handle both.

In [ ]:
# Working-directory configuration:
# Set the WORK_DIR environment variable (or override here) to point at
# the local or networked folder that holds the raw cohort data.
# Default Colab convention: the mounted Drive root.
import os
WORK_DIR = os.environ.get('WORK_DIR', '/content/drive/MyDrive')
print(f'WORK_DIR = {WORK_DIR}')


In [ ]:
# Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Step 2: Install TCIA download utilities
!pip install -q tcia_utils

In [ ]:
# Step 3: Create destination directory
import os
dest_dir = f'{WORK_DIR}/autopet_iii'
os.makedirs(dest_dir, exist_ok=True)

# Check available Drive space
import shutil
total, used, free = shutil.disk_usage('/content/drive')
print(f'Google Drive free space: {free / (1024**3):.1f} GB')
print(f'Destination: {dest_dir}')

In [ ]:
# Step 4: Parse the manifest to get series UIDs
# Defensive against tcia_utils API drift — column names like TotalSizeInBytes,
# ImageCount, etc. have changed across versions. The only column we strictly
# need is SeriesInstanceUID; everything else is reported best-effort.
from tcia_utils import nbia

series_df = nbia.getSeries(collection='PSMA-PET-CT-Lesions', format='df')
print(f'Total series: {len(series_df)}')
print(f'Columns available: {list(series_df.columns)}')

# Required: SeriesInstanceUID
if 'SeriesInstanceUID' not in series_df.columns:
    raise RuntimeError('SeriesInstanceUID column missing — TCIA API contract broken')

# Optional: modality breakdown
if 'Modality' in series_df.columns:
    print(f'Modalities: {series_df["Modality"].value_counts().to_dict()}')

# Optional: size — try common column names across tcia_utils versions
size_col = next((c for c in ('TotalSizeInBytes', 'TotalSize', 'FileSize', 'SeriesSize')
                 if c in series_df.columns), None)
if size_col:
    total_gb = series_df[size_col].sum() / (1024**3)
    print(f'Total size ({size_col}): {total_gb:.1f} GB')
else:
    print('Size column not present in this tcia_utils version — proceeding without size estimate.')
    print('(Original AutoPET-III is documented as ~117 GB across 1,791 series.)')

# Optional: image count (proxy for size)
img_col = next((c for c in ('ImageCount', 'NumberOfImages', 'InstanceCount')
                if c in series_df.columns), None)
if img_col:
    print(f'Total images ({img_col}): {int(series_df[img_col].sum()):,}')

In [ ]:
# Step 5: Download all series as .zip directly to Drive (no extraction)
#
# CHANGED 2026-04-26: previously this cell extracted each series into a per-UID
# directory of ~300-500 small .dcm files. Drive's FUSE driver makes a separate
# metadata API call per file create (~50-200 ms each), so extraction was
# costing ~40 s/series of pure overhead independent of the actual TCIA download.
# Now we save each series as a single {uid}.zip on Drive — one large sequential
# write per series. The downstream AutoPET-III processing notebook will read
# directly from these zips (same pattern as process_autopet_i.ipynb Step 4).
# Parallel workers exploit Drive's tolerance of concurrent single-file writes
# and TCIA's tolerance of moderate parallelism. Net speedup: ~3-5x.
#
# RESUME COMPATIBILITY: _download_log.txt format unchanged (one UID per line).
# Series previously downloaded as extracted directories are recognised by
# Step 7 verification — they do not need to be re-downloaded or re-zipped.

import time
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

NUM_WORKERS = 4  # TCIA tolerates this; raise to 6-8 only if no HTTP 429 seen
CHUNK_SIZE = 4 * 1024 * 1024  # 4 MB streaming chunks

series_uids = series_df['SeriesInstanceUID'].tolist()
total = len(series_uids)

log_path = f'{dest_dir}/_download_log.txt'
already_done = set()
if os.path.exists(log_path):
    with open(log_path) as f:
        already_done = set(line.strip() for line in f if line.strip())
    print(f'Resuming: {len(already_done)} series already downloaded')

to_download = [uid for uid in series_uids if uid not in already_done]
print(f'To download: {len(to_download)}/{total}')
print(f'Workers: {NUM_WORKERS}, chunk size: {CHUNK_SIZE // (1024*1024)} MB')
print()

log_lock = threading.Lock()
state_lock = threading.Lock()
state = {'downloaded': 0, 'failed': 0, 'bytes': 0}
failed = []

def download_one(uid):
    try:
        url = f'https://services.cancerimagingarchive.net/nbia-api/services/v1/getImage?SeriesInstanceUID={uid}'
        resp = requests.get(url, stream=True, timeout=300)
        resp.raise_for_status()

        # Stream zip bytes directly to Drive — single sequential write per series
        zip_path = f'{dest_dir}/{uid}.zip'
        n_bytes = 0
        with open(zip_path, 'wb') as f:
            for chunk in resp.iter_content(chunk_size=CHUNK_SIZE):
                if chunk:
                    f.write(chunk)
                    n_bytes += len(chunk)

        # Atomically append to log only after successful write
        # (interrupted writes leave a partial .zip but no log entry,
        # so the next resume re-downloads cleanly)
        with log_lock:
            with open(log_path, 'a') as f:
                f.write(uid + '\n')

        with state_lock:
            state['downloaded'] += 1
            state['bytes'] += n_bytes
            n_done = state['downloaded']
            if n_done % 25 == 0:
                done_total = n_done + len(already_done)
                rate_mb_s = state['bytes'] / (1024**2) / max(time.time() - start, 1)
                print(f'  Progress: {done_total}/{total} '
                      f'({n_done} new this session, {state["failed"]} failed) | '
                      f'{rate_mb_s:.1f} MB/s effective')
        return True
    except Exception as e:
        with state_lock:
            state['failed'] += 1
        return (uid, str(e))

start = time.time()
with ThreadPoolExecutor(max_workers=NUM_WORKERS) as ex:
    futures = {ex.submit(download_one, uid): uid for uid in to_download}
    for fut in as_completed(futures):
        result = fut.result()
        if result is not True:
            failed.append(result)

elapsed = time.time() - start
rate_per_min = state['downloaded'] / max(elapsed / 60, 1)
print(f'\nDone: {state["downloaded"]} downloaded, {state["failed"]} failed in {elapsed/60:.1f} min')
print(f'Throughput: {rate_per_min:.1f} series/min, {state["bytes"] / (1024**3):.1f} GB total')

if failed:
    failed_path = f'{dest_dir}/_failed.txt'
    with open(failed_path, 'w') as f:
        for uid, err in failed:
            f.write(f'{uid}\t{err}\n')
    print(f'Failed UIDs saved to {failed_path}')
    print(f'First 5 failures:')
    for uid, err in failed[:5]:
        print(f'  {uid}: {err[:120]}')

In [ ]:
# Step 6: Retry any failed downloads
# Re-running Step 5 cell will skip already-logged successes and retry the rest,
# which includes any that were in _failed.txt from this session.
# If TCIA returned HTTP 429 (rate limited), reduce NUM_WORKERS in Step 5 to 2 before retry.
if os.path.exists(f'{dest_dir}/_failed.txt'):
    with open(f'{dest_dir}/_failed.txt') as f:
        retry_uids = [line.split('\t')[0] for line in f if line.strip()]
    print(f'{len(retry_uids)} series in _failed.txt')
    print('Re-run Step 5 to retry — the log file will skip already-done series.')
else:
    print('No failures to retry')

In [ ]:
# Step 7: Sanity check — count + missing UIDs only (no Drive traversal)
# Step 5's atomic write pattern (zip write → log entry, under lock) guarantees
# every logged UID has a corresponding file on Drive. Verification only needs
# to compare manifest vs log; exhaustive file-system checks are not required.
with open(log_path) as f:
    done = set(line.strip() for line in f if line.strip())
missing = set(series_uids) - done
print(f'Log: {len(done)}/{len(series_uids)}, missing: {len(missing)}')
if missing:
    print(f'Missing UIDs: {sorted(missing)}')
    print('Re-run Step 5 to fetch — resume logic will only attempt these.')
else:
    print('Download complete.')